# PLANIFICADORES

OBJETIVO: Obtener el minimo local
Ajustamos el learning rate para llegar cuanto antes:
- Muy grande -> nos pasamos
- Muy pequeño -> no llegamos nunca
Podemos utilizar Planificadores (= condiciones que se verifican en la ejecución) para poder ajustar mejor



EXAMEN: "ultima parte y esta"

## Introducción
Descenso por gradiente: dado uno parametros iniciales, minimiza la perdida de manera iterativa, moviendose en la direccion de descenso mas prnunciada, la dada por el neg-gradiente de la perdida calculada en cada paso (batch)

PLANIFICADOR: func. para determinar el learning rate en función de t(num de batches procesados)

FACTOR DE APRENDIZAJE CONSTANTE: SGD puede no converger o hacerlo muy lentamente. Los planificadores nos permiten variar / jugar con estos parámetros para mejorar el proceso de aprendizaje





Heredan de una clase base.
Parámetros:
- "optimizer": optimizador cuyo learning rate se ajusta
- "last_epoch": indice de la ultima epcoa vista por el planificador

Hay distintos tipos de schedulers:
- StepLR: cambia el learning rate cada X epocas(CAÍDA ESCALONADA)
- ReduceLROnPlateau: cambia el learning rate si en X epocas (patience) no hemos cumplido una mejora esperada (threshold)
- CosineAnnealing: reduce el leasrning rate aplicando una funcion coseno
- Warmup: añadido que incrementa el learning rate rápidamente los primeros "warmpu_t" pasos para luego dejar a la función decrementar el learning rate a su manera
- Warmup-Stable-Decay: scheduler con un warmup inicial("num_warmup_steps"), tras el cual hay una fase de estabilización (el learning rate no varía) de "num_stable_steps", para luego pasar al decrecimiento ("num_decay_steps")

In [ ]:
import numpy as np; import matplotlib.pyplot as plt
import torch; import torch.optim as optim; import torch.nn as nn

ModuleNotFoundError: No module named 'numpy'

In [ ]:
def get_lrs(optimizer, scheduler, steps=100, val_loss=None):
    lrs = []
    for t in range(steps):
        lrs += scheduler.get_last_lr(); optimizer.step()
        if val_loss == None: scheduler.step()
        else: scheduler.step(val_loss[t])
    return lrs

In [ ]:
# Definimos un modelo. Para cada epoca PAR, se reduce el learning rate a la mitadP
model = nn.Linear(10, 2)
optimizer = optim.SGD(model.parameters(), lr=0.1)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)
steps = 10
lrs = get_lrs(optimizer, scheduler, steps=steps)
_, ax = plt.subplots(figsize=(10, 1.75))
ax.stairs(lrs, linewidth=2.5)
ax.grid()
ax.set(xlim=(-0.1, steps + 0.1), xticks=np.arange(0, steps + 1))

ReduceLROnPlateau
# Parametros
#   patience, threshold(mejora minima que consideramos valida) -> permiten establecer una mejora esperada que, en caso de no cumplirse en X epocas, reducimos el learning rate
#   mode = 'min' -> Buscamos minimizar el parametro; 'max' -> Buscamos maximizar el parametro

In [ ]:
model = nn.Linear(10, 2)
optimizer = optim.SGD(model.parameters(), lr=0.1)
steps = 30
x = np.linspace(0, steps + 1)
exp_lambda = 0.1
val_loss = torch.from_numpy(exp_lambda * np.exp(-exp_lambda * x))
val_loss += (
    val_loss
    * (
        torch.rand_like(
            val_loss,
        )
        - 0.5
    )
    * 0.5
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=2)
lrs = get_lrs(optimizer, scheduler, steps=steps, val_loss=val_loss)
_, (ax_loss, ax) = plt.subplots(2, figsize=(10, 2), sharex=True)
ax_loss.plot(x, val_loss)
ax.stairs(lrs, linewidth=2.5)
ax.grid()
ax.set(xlim=(-0.1, steps + 0.1), xticks=np.arange(0, steps + 1, 5))


# Seguimos la funcion de perdida. Cuando hay una llanura, reducimos para poder reducir más
# Cuando realizamos un experimento preliminar, utilizamos este

CosineAnnealing: aplica una funcion coseno
# Parametros
#   T-max = numero max de pasos
#   eta_min = learning rate minimo

In [ ]:
model = nn.Linear(10, 2)
optimizer = optim.SGD(model.parameters(), lr=0.1)
steps = 50
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, steps, eta_min=0.010)
lrs = get_lrs(optimizer, scheduler, steps=steps)
_, ax = plt.subplots(figsize=(10, 2))
ax.stairs(lrs, linewidth=2.5)
ax.grid()
ax.set(xlim=(-0.1, steps + 0.1), xticks=np.arange(0, steps + 1, 5))

Warmup: scheduler donde al principio aumentamos el learning rate (no se utilizan tanto)
# Parametros:
#   t_initial= nimero total de epocas/pasos
#   warmup_t = pasos de warmup inicial

In [ ]:
from timm.scheduler.cosine_lr import CosineLRScheduler

model = nn.Linear(10, 2)
optimizer = optim.SGD(model.parameters(), lr=0.1)
steps = 50
scheduler = CosineLRScheduler(
    optimizer, t_initial=steps, warmup_t=0, warmup_lr_init=1e-2
)
lrs = [scheduler._get_lr(t)[0] for t in range(steps)]
_, ax = plt.subplots(figsize=(10, 2))
ax.stairs(lrs, linewidth=2.5)
ax.grid()
ax.set(xlim=(-0.1, steps + 0.1), xticks=np.arange(0, steps + 1, 5))

Warmup-Stable-Decay:

In [ ]:
from pytorch_optimizer import get_wsd_schedule

model = nn.Linear(10, 2)
optimizer = optim.SGD(model.parameters(), lr=0.1)
steps = 50
scheduler = get_wsd_schedule(optimizer, 5, 10, 35)
lrs = get_lrs(optimizer, scheduler, steps=steps)
_, ax = plt.subplots(figsize=(10, 2))
ax.stairs(lrs, linewidth=2.5)
ax.grid()
ax.set(xlim=(-0.1, steps + 0.1), xticks=np.arange(0, steps + 1, 5))

# Ejemplo

In [ ]:
def exp(model, device, train_loader, test_loader, optimizer, scheduler, epochs=15):
    loss_fn = torch.nn.CrossEntropyLoss()
    for epoch in range(epochs):
        print(f'Epoch {epoch}: train:', end=' ')
        model.train(); trsize, trbatches, trloss, tracc = 0, 0, 0, 0
        for batch in train_loader:
            X, y = batch['X'].to(device), batch['y'].to(device); trsize += len(X); trbatches += 1
            pred = model(X); loss = loss_fn(pred, y); trloss += loss.item()
            tracc += (pred.argmax(1) == y).type(torch.float).sum().item()
            loss.backward(); optimizer.step(); optimizer.zero_grad(); scheduler.step()
        trloss /= trbatches; tracc /= trsize
        print(f'loss {trloss:g} acc {tracc:.2%} test:', end=' ')
        model.eval(); tesize, tebatches, teloss, teacc = 0, 0, 0, 0
        with torch.no_grad():
            for batch in test_loader:
                X, y = batch['X'].to(device), batch['y'].to(device); tesize += len(X); tebatches += 1
                pred = model(X); teloss += loss_fn(pred, y).item()
                teacc += (pred.argmax(1) == y).type(torch.float).sum().item()
        teloss /= tebatches; teacc /= tesize
        print(f'loss {teloss:g} acc {teacc:.2%}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import timm


device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)
device
model_name = "resnet50.fb_swsl_ig1b_ft_in1k"  # avg_top1 71.5 param_count 25.56
model = timm.create_model(model_name, pretrained=True, num_classes=10).to(device)

In [ ]:
# Va reduciendo el learningn rate siguiendo una funcion de coseno
# En ingun momento se nos estanca el modelo.

import datasets
from torch.utils.data import DataLoader

ds = (
    datasets.load_dataset("uoft-cs/cifar10")
    .with_format("torch")
    .rename_columns({"img": "X", "label": "y"})
)


def datanorm(example):
    example["X"] = example["X"].to(torch.float) / 255.0
    return example


ds = ds.map(datanorm, batched=True)
train_ds = ds["train"].to_iterable_dataset(num_shards=1024)
test_ds = ds["test"].to_iterable_dataset(num_shards=1024)
train_loader = DataLoader(train_ds, batch_size=32)
test_loader = DataLoader(test_ds, batch_size=32)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3
)
steps = 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, steps, eta_min=1e-5)
exp(model, device, train_loader, test_loader, optimizer, scheduler, epochs=steps)

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/120M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/23.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Epoch 0: train: loss 1.90708 acc 36.96% test: loss 3.32967 acc 32.49%
Epoch 1: train: 

In [ ]:
# MNIST
import numpy as np
import matplotlib.pyplot as plt
import torch
import timm

device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)
device
model_name = "resnet50.fb_swsl_ig1b_ft_in1k"  # avg_top1 71.5 param_count 25.56
model = timm.create_model(model_name, pretrained=True, num_classes=10, in_chans=1).to(
    device
)

In [ ]:
import datasets
from torch.utils.data import DataLoader

ds = (
    datasets.load_dataset("ylecun/mnist")
    .with_format("torch")
    .rename_columns({"image": "X", "label": "y"})
)


def datanorm(example):
    example["X"] = example["X"].to(torch.float) / 255.0
    return example


ds = ds.map(datanorm, batched=True)
train_ds = ds["train"].to_iterable_dataset(num_shards=1024)
test_ds = ds["test"].to_iterable_dataset(num_shards=1024)
train_loader = DataLoader(train_ds, batch_size=32)
test_loader = DataLoader(test_ds, batch_size=32)
# Definimos el optimizador
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3
)
steps = 20
# Definimos el scheduler que irá modificando el learning rate: CosineAnnealing
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, steps, eta_min=1e-5)
exp(model, device, train_loader, test_loader, optimizer, scheduler, epochs=steps)

In [ ]:
# Fashion MNIST
device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)
device
model_name = "resnet50.fb_swsl_ig1b_ft_in1k"  # avg_top1 71.5 param_count 25.56
model = timm.create_model(model_name, pretrained=True, num_classes=10, in_chans=1).to(
    device
)

In [ ]:
import datasets
from torch.utils.data import DataLoader

ds = (
    datasets.load_dataset("zalando-datasets/fashion_mnist")
    .with_format("torch")
    .rename_columns({"img": "X", "label": "y"})
)


def datanorm(example):
    example["X"] = example["X"].to(torch.float) / 255.0
    return example


ds = ds.map(datanorm, batched=True)
train_ds = ds["train"].to_iterable_dataset(num_shards=1024)
test_ds = ds["test"].to_iterable_dataset(num_shards=1024)
train_loader = DataLoader(train_ds, batch_size=32)
test_loader = DataLoader(test_ds, batch_size=32)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3
)
steps = 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, steps, eta_min=1e-5)
exp(model, device, train_loader, test_loader, optimizer, scheduler, epochs=steps)

In [ ]:
""" PRUEBA DE TODOS LOS PLANIFICADORES CON MNIST """
####################### Primero cargamos el modelo
import numpy as np
import matplotlib.pyplot as plt
import torch
import timm
#Establecemos el dispositivo...
device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)
device
# Y creamos el modelo en el dispositivo
model_name = "resnet50.fb_swsl_ig1b_ft_in1k"  # avg_top1 71.5 param_count 25.56
model = timm.create_model(model_name, pretrained=True, num_classes=10, in_chans=1).to(device)

######################### Segundo: cargamos el dataset:
import datasets
from torch.utils.data import DataLoader

ds = (
    datasets.load_dataset("ylecun/mnist")
    .with_format("torch")
    .rename_columns({"image": "X", "label": "y"})
)
# Definimos la funcion que utilizaremos para obtener y transformar los datos...
def datanorm(example):
    example["X"] = example["X"].to(torch.float) / 255.0
    return example
# Obtenemos los datos, guardandolos en batches y clasificanolos segun training y testing
ds = ds.map(datanorm, batched=True)
train_ds = ds["train"].to_iterable_dataset(num_shards=1024)
test_ds = ds["test"].to_iterable_dataset(num_shards=1024)
train_loader = DataLoader(train_ds, batch_size=32)
test_loader = DataLoader(test_ds, batch_size=32)


# Definimos el optimizador (siempre es AdamW)
# IMPORTANTE: este optimizador actualizará TODOS los parámetros que tengan requires_grad = True. Originalemnte, estos son todos los parámetros de la red
#             Si quisiseramos que SOLO entrenara la cabeza, se debería "congelar" el cuerpo antes.
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3
)

steps = 20

####################### Utilizaremos los distintos planificadores para entrenar el modelo
"""
LRScheduler: caída base.
- optimizer = optimizador cuyo lr se ajusta
- last_epoch = indice de la ultima epoca vista por el planificador

StepLR: cambia el learning rate cada X epocas(CAÍDA ESCALONADA)
- optimizer
- step_size: cuantas epocas deben pasar antes de aplicar la reduccion
- gamma: factor de reduccion del lr
- last_epoch

## PLANIFICADOR IDEAL PARA CUANDO EL ENTRENAMIENTO DEBE REALIZRSE DURANTE MUCHAS EPOCAS Y SE OBSERVA QUE LA PERDIDA SE ESTANCA PERIODICAMENTE,
   AUNQUE NO SABEMOS EXACTAMENTE EN QUE PUNTO. Multiplica el LR por un factor gamma constante cada X epocas

"""
#scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5, last_epoch=-1)

"""
ReduceLROnPlateau: cambia el lr dependiendo de la pérdida.
- mode: el lr se reduce cuando la metrica monitorizada deja de decrecer (min) o crecer (max)
- factor: factor de reduccion de lr
- patience: num de pasos sin mejora tras los cuales se decide reducir el lr
- threshold: umbral minimo para considerar una mejora significativa
- threshold_mode: umbral relativo al mejor valor hallado(rel) o absoluto(abs)
- cooldown: num de pasos a esperar para seguir con el proceso tras una reduccion del lr
- min_lr: cota inferior del lr. La minima a la que podemos reducirlo
- eps: si la diff entre el nuevo lr y el anterior es < a eps, lr NO SE ACTUALIZA
"""
#scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, threshold=0.0001, threshold_mode='rel', cooldown=0, min_lr=0, eps=1e-08, verbose=False)


"""
CosineAnnealing: reduce el LR segun la funcion coseno
- T_max: max numero de pasos/epocas para llegar al lr mínimo
- eta_min: learnign rate minimo
"""
#scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=0.001, last_epoch=-1)

"""
CosineAnnealing + linear warmup: introduce un warmup al incicio que hace crecer el LR
- t_initial: numero total de pasos
- warmup_t: pasos del warmup inicial. Si warmup_t = 0, sería igual al Cosi


"""

#scheduler = torch.optim.lr_scheduler.CosineLRScheduler(optimizer, t_initial=steps, warmup_t = 5, warmup_lr_init=1e-2)

"""
Warmup-Stable-Decay: planificador estructurado en 3 fases; 1) warmup, 2) estabilización 3) decay
- num_warmup_steps: numero de pasos de warmup
- num_stable_steps: numero de pasos de estabilización
- num_decay_steps: numero de pasos de decay
"""
from pytorch_optimizer import get_wsd_schedule
scheduler = get_wsd_schedule(optimizer, num_warmup_steps=5, num_stable_steps=10, num_decay_steps=35)


exp(model, device, train_loader, test_loader, optimizer, scheduler, epochs=steps)



In [ ]:
"""
Manera de congelar el cuerpo de la red para entrenar solo la cabeza:

Antes de crear el optimizador:

for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3
)



//////////////////////////
for p in model.parameters():
    p.requires_grad = False
for p in model.fc.parameters():
    p.requires_grad = True
exp(model, device, train_loader, test_loader, optimizer, scheduler, epochs=steps)


"""